# Stage 18 · Export the three matched personal-scene videos

Attach the private **personal-input.zip** and **personal-s42-stage17-review.zip** (intact or expanded). Select **GPU T4 ×2** and **Run All**. Setup verifies three 75,000-update models. They follow the same interpolated path through the 12 registered test poses at 240 × 320. Each clip contains 120 RGB frames at 24 fps.

Stage 17 scoring took 9.2–9.7 seconds per view. At those rates 360 frames on two GPUs project to 28–29 minutes. Allow **30–45 minutes** with setup, caching and encoding. Save **personal-s42-stage18-cache.zip** at the halfway link. In a fresh session attach the cache alongside the two inputs. Send **personal-s42-stage18-videos.zip** privately after all three movies report 120 decoded frames. On interruption send the latest cache and traceback.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from contextlib import ExitStack
import base64
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
import torch
from PIL import Image
from IPython.display import HTML, display

inputs, outputs = Path("/kaggle/input"), Path("/kaggle/working")
work = outputs / "personal-stage18"
work.mkdir(exist_ok=True, parents=True)
if "stage18_temp" not in globals():
    stage18_temp = TemporaryDirectory(prefix="personal-video-inputs-")
temp = Path(stage18_temp.name)
arms = ('dense-vanilla', 'sparse-vanilla', 'sparse-regularized')
checkpoint_hashes = {'dense-vanilla': 'f5c8c8251442b43717be7f519e3321afd443b8adb8fa08b3d6d094e64b2acb33', 'sparse-vanilla': 'ad840a886b76d7e6115ead08cbfc7953c66aa06f753bc72aac3840d0110e301f', 'sparse-regularized': '5748f503c696d3eacdfaf66cba24f7261e695a07624da59eb510d25c361978ec'}
model_hashes = {'dense-vanilla': '85fd7bcac9b755331c6d056674319532121ad287a3483b27bba8151250c48274', 'sparse-vanilla': '8080987713d26ac100270118036bbfea07212f46422bf73d145da5e105666254', 'sparse-regularized': '562662d5c22a1964a7c53f3d568e43e1c44b62e8779a0e1000121b80a18ad9dc'}
data_hash = 'd39c6ff67262904660cf715ea3ba6cdbd17ae3a0cf07a1b83a917cbb50dbe8ae'
score_hash = '2c7f7d983bd7eabc168279d25efd7aa9aea7b19a4af3e28400e8c57a5d7e53b0'
frames, fps, chunk = 120, 24, 1024
if torch.cuda.device_count() < 2:
    raise RuntimeError("Choose GPU T4 ×2 and restart the notebook.")
visible = os.environ.get("CUDA_VISIBLE_DEVICES")
devices = [v.strip() for v in visible.split(",")][:2] if visible else ["0", "1"]
assert len(set(devices)) == 2
print("Rendering on", [torch.cuda.get_device_name(i) for i in range(2)])

def read(path):
    return json.loads(path.read_bytes())

def sha(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()

def unpack(path):
    dest = temp / hashlib.sha256(str(path).encode()).hexdigest()
    dest.mkdir(exist_ok=True)
    with zipfile.ZipFile(path) as z:
        assert len(z.namelist()) == len(set(z.namelist()))
        for name in z.namelist():
            assert not Path(name).is_absolute() and ".." not in Path(name).parts
            z.extract(name, dest)
    return dest

def source(marker, filename, check):
    for p in sorted(inputs.rglob(marker)):
        if check(p.parent):
            return p.parent
    for p in sorted(inputs.rglob(filename)):
        folder = unpack(p)
        if check(folder):
            return folder
    raise FileNotFoundError(f"Attach the reviewed {filename} or its expanded dataset.")

repo = source("RECOVERY_PERSONAL.txt", "personal-input.zip", lambda d: read(d / "RECOVERY_PERSONAL.txt")["data_hash"] == data_hash)
sys.path.insert(0, str(repo))
from recovery.data import verify_manifest
from recovery.media import digest, model_fingerprint, video, write_json
from recovery.training import implementation_hash, load_final
from sparse_nerf.video import encoder_executable

assert implementation_hash() == "4b85a7f9d5836739fb562c5fb3ff25fa68b063073bbea539e38ed1130f088eb5"
assert hashlib.sha256((repo/"recovery/media.py").read_text(encoding="utf-8").encode()).hexdigest() == "fd13945accd1ec6938b7ebceccec31fe1f8716dc2a2ad51d84068d560e09a3af"
def complete(folder, arm):
    cp = folder / "runs/personal" / arm / "seed-42/checkpoint.pt"
    return cp.is_file() and sha(cp) == checkpoint_hashes[arm]

pair = source("stage17-review.json","personal-s42-stage17-review.zip",lambda d: all(complete(d,a) for a in arms) and read(d/"stage17-review.json")["scored"])
manifest = work / "personal.json"
data = verify_manifest(pair/"manifests/personal.json", repo/"data/personal")
assert data["data_hash"] == data_hash
write_json(manifest, data)
trajectory = dict(data_hash=data_hash, downscale=4, frames=frames, fps=fps, path="capture-interpolation-v1")
trajectory_hash = digest(trajectory)
runs, video_dirs, descriptors = {}, {}, {}
for arm in arms:
    path = pair / "runs/personal" / arm / "seed-42"
    meta, metrics = read(path/"run.json"), read(path/"eval/metrics.json")
    assert (meta["protocol"], meta["seed"], meta["scene"], meta["downscale"]) == (arm, 42, "personal", 4)
    assert metrics["run_id"] == meta["run_id"] and metrics["comparison_hash"] == digest(metrics["comparison"]) == score_hash
    assert metrics["comparison"]["training_config"] == meta["training_config"] and meta["training_config"]["steps"] == 75000
    model, _ = load_final(path, manifest, torch.device("cpu"))
    assert model_fingerprint(model) == metrics["model_sha256"] == model_hashes[arm]
    del model
    # Only frame caches and encodes need a writable location.
    runs[arm] = path
    video_dirs[arm] = work/arm/"video"/trajectory_hash[:12]
    (video_dirs[arm]/"frames").mkdir(parents=True, exist_ok=True)
    desc = descriptors[arm] = dict(run_id=meta["run_id"], model_sha256=model_hashes[arm],
                                    trajectory=trajectory, trajectory_hash=trajectory_hash)
    desc_path = video_dirs[arm]/"trajectory.json"
    if desc_path.exists():
        assert read(desc_path) == desc
    else:
        assert not any((video_dirs[arm]/"frames").glob("*.png"))
        write_json(desc_path, desc)
    print(f"Verified {arm}: 75,000 updates, {model_hashes[arm][:12]}")
ffmpeg = encoder_executable()
encoders = subprocess.check_output([ffmpeg, "-hide_banner", "-encoders"], text=True)
assert "libx264" in encoders

In [ ]:
def frame_path(arm, i):
    return video_dirs[arm]/"frames"/f"{i:05d}.png"

def valid_frame(path):
    if not path.is_file():
        return False
    try:
        with Image.open(path) as im:
            if im.format != "PNG" or im.size != (240, 320) or im.mode != "RGB":
                return False
            im.verify()
        return True
    except (OSError, SyntaxError, ValueError):
        return False

def frame_name(arm, i):
    return f"{arm}/frames/{i:05d}.png"

def encoding_record(arm, movie):
    return dict(trajectory=descriptors[arm], frames=[sha(frame_path(arm, i)) for i in range(frames)], movie=sha(movie))

session_timing = {}
allowed = {frame_name(a, i): frame_path(a, i) for a in arms for i in range(frames)}
for arm in arms:
    allowed[f"{arm}/trajectory.json"] = video_dirs[arm]/"trajectory.json"
    allowed[f"{arm}/novel.mp4"] = video_dirs[arm]/"novel.mp4"

def restore(folder):
    record = read(folder/"stage18-review.json")
    assert record["checkpoints"] == checkpoint_hashes and record["trajectory"] == trajectory
    assert set(record["sha256"]).issubset(allowed)
    for arm in arms:
        key = f"{arm}/trajectory.json"
        assert key in record["sha256"] and read(folder/key) == descriptors[arm]
    for name, expected in record["sha256"].items():
        src, dst = folder/name, allowed[name]
        assert sha(src) == expected
        if name.endswith("trajectory.json"):
            continue
        if name.endswith(".png"):
            assert valid_frame(src)
        if dst.is_file() and sha(dst) == expected:
            continue
        if dst.is_file() and name.endswith(".png") and valid_frame(dst):
            raise ValueError(f"Two readable cached renders disagree for {name}")
        tmp = dst.with_suffix(".restoring")
        shutil.copyfile(src, tmp)
        tmp.replace(dst)
    if record.get("completed"):
        for arm in arms:
            assert record["frame_indices"][arm] == list(range(frames))
            names = [frame_name(arm,i) for i in range(frames)]
            # Bind a restored movie only to the exact frame bytes in its original receipt.
            expected = dict(trajectory=descriptors[arm], frames=[record["sha256"][n] for n in names],
                            movie=record["sha256"][f"{arm}/novel.mp4"])
            write_json(video_dirs[arm]/"encoding.json", expected)
    timing = record.get("last_timed_render_segment", {})
    if timing.get("end_count",0) > session_timing.get("end_count",0):
        session_timing.update(timing)

for p in sorted(inputs.rglob("stage18-review.json")):
    restore(p.parent)
for filename in ("personal-s42-stage18-cache.zip", "personal-s42-stage18-videos.zip"):
    for p in sorted(inputs.rglob(filename)):
        restore(unpack(p))

def cached():
    return {a: [i for i in range(frames) if valid_frame(frame_path(a, i))] for a in arms}

def snapshot(final=False):
    indices = cached()
    files = {}
    for arm, ids in indices.items():
        names = [f"{arm}/trajectory.json", *(frame_name(arm, i) for i in ids)]
        if final:
            assert len(ids) == frames
            names.append(f"{arm}/novel.mp4")
        files.update((name, allowed[name]) for name in names)
    record = dict(trajectory=trajectory, trajectory_hash=trajectory_hash, checkpoints=checkpoint_hashes,
                  models=model_hashes, frame_indices=indices, completed=final, last_timed_render_segment=session_timing,
                  sha256={n: sha(p) for n,p in files.items()})
    dest = outputs / ("personal-s42-stage18-videos.zip" if final else "personal-s42-stage18-cache.zip")
    tmp = dest.with_suffix(".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as z:
        for name,p in files.items():
            z.write(p, name)
        z.writestr("stage18-review.json", json.dumps(record, indent=2, allow_nan=False))
    with zipfile.ZipFile(tmp) as z:
        assert z.testzip() is None and set(z.namelist()) == set(files) | {"stage18-review.json"}
    os.replace(tmp, dest)
    print(f"Saved {dest.name}: {sum(map(len, indices.values()))} frames.", flush=True)
    return dest

def download(path):
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    display(HTML(f'<a download="{path.name}" href="data:application/zip;base64,{encoded}">Download {path.name}</a>'))

print("Reusable trajectory frames:", {a: len(ids) for a, ids in cached().items()})

In [ ]:
worker = """
import json
from pathlib import Path
import sys
import time
import torch
from PIL import Image
from recovery.training import load_final
from recovery.media import model_fingerprint, render_maps, rgb_uint8
from sparse_nerf.data import SceneDataset

def render(jobs, manifest, frames, chunk, device):
    for job in jobs:
        run, dest = Path(job["run"]), Path(job["dest"])
        model, meta = load_final(run, manifest, device)
        desc = json.loads((dest/"trajectory.json").read_text(encoding="utf-8"))
        assert model_fingerprint(model) == desc["model_sha256"] and meta["run_id"] == desc["run_id"]
        dataset = SceneDataset(manifest, "test", meta["downscale"])
        for i in job["indices"]:
            start = time.monotonic()
            rgb, _, _ = render_maps(model, dataset, dataset.video_rays(i, frames),
                                    meta["protocol"] == "sparse-regularized", device, chunk)
            path = dest/"frames"/f"{i:05d}.png"
            tmp = path.with_suffix(".pending")
            Image.fromarray(rgb_uint8(rgb)).save(tmp, format="PNG")
            tmp.replace(path)
            print(f"{meta['protocol']} frame {i+1}/{frames} in {time.monotonic()-start:.2f}s", flush=True)
        del model

if __name__ == "__main__":
    config = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
    render(config["jobs"], Path(config["manifest"]), config["frames"], config["chunk"], torch.device("cuda:0"))
"""

worker_path = temp/"render_capture.py"
worker_path.write_text(worker, encoding="utf-8")

def render_missing():
    have = cached()
    pending = [(a, i) for a in arms for i in range(frames) if i not in have[a]]
    if not pending:
        print("All 360 trajectory frames are present.")
        return
    total = frames*len(arms)
    start_count = total-len(pending)
    damaged = {(a,i) for a,i in pending if frame_path(a,i).exists()}
    last_save = start_count
    started = time.monotonic()
    processes = []
    half_link = False
    with ExitStack() as stack:
        try:
            for lane, gpu in enumerate(devices):
                assigned = pending[lane::len(devices)]
                if not assigned:
                    continue
                jobs = [dict(run=str(runs[a]), dest=str(video_dirs[a]), indices=[i for arm,i in assigned if arm == a])
                        for a in arms if any(arm == a for arm,_ in assigned)]
                config = temp/f"lane-{lane}.json"
                write_json(config, dict(jobs=jobs, manifest=str(manifest), frames=frames, chunk=chunk))
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, OMP_NUM_THREADS="1", MKL_NUM_THREADS="1",
                           PYTHONPATH=os.pathsep.join([str(repo), os.environ.get("PYTHONPATH", "")]))
                log = work/f"gpu-{lane}.log"
                writer = stack.enter_context(log.open("ab"))
                reader = stack.enter_context(log.open(encoding="utf-8", errors="replace"))
                reader.seek(0, os.SEEK_END)
                proc = subprocess.Popen([sys.executable, "-u", str(worker_path), str(config)], cwd=repo,
                                        env=env, stdout=writer, stderr=subprocess.STDOUT)
                processes.append((proc, reader))
            while True:
                done = True
                for lane,(proc, reader) in enumerate(processes):
                    code = proc.poll()
                    for line in reader:
                        print(f"[GPU {lane}] {line.rstrip()}", flush=True)
                    if code not in (0,None):
                        raise RuntimeError(f"GPU worker {lane} exited with status {code}.")
                    done = done and code == 0
                # Workers publish complete PNGs using atomic rename. Only newly assigned
                # paths count toward throughput; a full PNG check precedes each export.
                count = start_count + sum(valid_frame(frame_path(a,i)) if (a,i) in damaged
                                          else frame_path(a,i).is_file() for a,i in pending)
                if count-last_save >= 30 or done:
                    session_timing.update(start_count=start_count, end_count=count, seconds=time.monotonic()-started)
                    archive = snapshot()
                    if not half_link and count >= total//2:
                        download(archive)
                        half_link = True
                    if count > start_count:
                        seconds = (time.monotonic()-started)/(count-start_count)
                        print(f"{count}/{total} frames; about {(total-count)*seconds/60:.1f} min of rendering left.", flush=True)
                    last_save = count
                if done:
                    break
                time.sleep(4)
        finally:
            for proc,_ in processes:
                if proc.poll() is None:
                    proc.terminate()
            for proc,_ in processes:
                proc.wait()
    assert all(len(ids) == frames for ids in cached().values())

try:
    render_missing()
except BaseException:
    download(snapshot())
    raise

In [ ]:
def decoded_count(movie):
    result = subprocess.run([ffmpeg, "-v", "error", "-xerror", "-nostdin", "-nostats", "-progress", "pipe:1",
                             "-i", str(movie), "-map", "0:v:0", "-an", "-f", "null", "-"],
                            capture_output=True, text=True, check=True)
    return int([s.partition("=")[2] for s in result.stdout.splitlines() if s.startswith("frame=")][-1])

assert all(len(ids) == frames for ids in cached().values())
for arm in arms:
    movie = video_dirs[arm]/"novel.mp4"
    receipt = video_dirs[arm]/"encoding.json"
    if movie.exists() and receipt.exists() and read(receipt) == encoding_record(arm, movie):
        print(f"Reusing verified {arm} encode.")
    else:
        # The frozen exporter verifies every cached frame and encodes the numbered PNGs.
        model, meta = load_final(runs[arm], manifest, torch.device("cpu"))
        assert model_fingerprint(model) == descriptors[arm]["model_sha256"]
        movie = video(work/arm, manifest, model, meta, torch.device("cpu"), chunk, frames=frames, fps=fps)
        del model
    assert decoded_count(movie) == frames
    write_json(receipt, encoding_record(arm, movie))
    print(arm, "120 decoded frames, 24 fps.", sha(movie))
download(snapshot(final=True))